In [1]:
import sympy as sp
import sys
sys.path.append('../')
import pathlib as pl
from SymEigen import *
from sympy import symbols
from project_dir import backend_source_dir
from affine_body_core import compute_J_vec

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")

In [2]:
kappa  = Eigen.Scalar("kappa")

# Basis vectors
ni_bar = Eigen.Vector("ni_bar", 3)
bi_bar = Eigen.Vector("bi_bar", 3)
bj_bar = Eigen.Vector("bj_bar", 3)
nj_bar = Eigen.Vector("nj_bar", 3)

# Affine Body DOF vectors
qi = Eigen.Vector("qi",12)
qj = Eigen.Vector("qj",12)



$$
E:=\frac{1}{2} K \sin^2(\theta- \tilde \theta) \\
  =\frac{1}{2}K\left(\sin \theta \cos \tilde\theta - \cos \theta \sin \tilde \theta \right)^2
$$

$$
\mathbf{F}_{01} = \begin{bmatrix}
\mathbf{\bar{\mathbf{b}}}_i \\ \mathbf{\bar{\mathbf{n}}}_i \\
\mathbf{\bar{\mathbf{b}}}_j \\ \mathbf{\bar{\mathbf{n}}}_j \\
\end{bmatrix}_{12 \times 1}
$$

Frame Affine Body Mapping:

$$
\mathbf{J}_{01} = \begin{bmatrix}
\mathbf{J}(\bar{\mathbf{b}}_i) & 0 \\
\mathbf{J}(\bar{\mathbf{n}}_i) & 0 \\
0 & \mathbf{J}(\bar{\mathbf{b}}_j) \\
0 & \mathbf{J}(\bar{\mathbf{n}}_j) \\
\end{bmatrix}_{12 \times 24}
$$

$$
\mathbf{F}_{01} = \mathbf{J}_{01} \cdot
\begin{bmatrix}
\mathbf{q}_i \\
\mathbf{q}_j \\
\end{bmatrix}
$$

In [3]:
# Mapping
J01 = sp.Matrix.zeros(12, 24)
J01[0:3, 0:12] = compute_J_vec(ni_bar)
J01[3:6, 0:12] = compute_J_vec(bi_bar)
J01[6:9, 12:24] = compute_J_vec(nj_bar)
J01[9:12, 12:24] = compute_J_vec(bj_bar)

content = ""

# from ABD q to F
F0_q = J01 @ sp.Matrix.vstack(qi, qj)
Cl_F01 = Gen.Closure(
    ni_bar,
    bi_bar,
    qi,  # Affine Body i
    nj_bar,
    bj_bar,
    qj,
)  # Affine Body j

# from F Gradient to ABD Gradient
G12 = Eigen.Vector("G12", 12)
J01T_G01 = J01.T @ G12
Cl_G01 = Gen.Closure(
    G12,
    ni_bar,
    bi_bar,  # Affine Body i
    nj_bar,
    bj_bar,
)  # Affine Body j
# from F Hessian to ABD Hessian
H12x12 = Eigen.Matrix("H12x12", 12, 12)
J01T_H01_J01 = J01.T @ H12x12 @ J01
Cl_H01 = Gen.Closure(
    H12x12,
    ni_bar,
    bi_bar,  # Affine Body i
    nj_bar,
    bj_bar,
)  # Affine Body j

content += f""" 
// Mapping between ABD qi qj to F01

{Cl_F01("F01_q", F0_q)}
{Cl_G01("J01T_G01", J01T_G01)}
{Cl_H01("J01T_H01_J01", J01T_H01_J01)}  

"""

F01_q = Eigen.Vector("F01_q", 12)
ni = F01_q[0:3, 0]
bi = F01_q[3:6, 0]
nj = F01_q[6:9, 0]
bj = F01_q[9:12, 0]

def compute_cos_theta(ni, nj, bi, bj):
    return (bi.dot(bj) + ni.dot(nj)) / 2


def compute_sin_theta(ni, nj, bi, bj):
    return (bi.dot(nj) - ni.dot(bj)) / 2


cos_theta = compute_cos_theta(ni, nj, bi, bj)
sin_theta = compute_sin_theta(ni, nj, bi, bj)

theta_tilde = Eigen.Scalar("theta_tilde")
cos_tilde = cos(theta_tilde[0, 0])
sin_tilde = sin(theta_tilde[0, 0])
# Angle
theta = atan2(sin_theta, cos_theta)
CL_Angle = Gen.Closure(F01_q)
content += f""" //Revolute Joint Angle

{CL_Angle("theta", theta)}
// -------------------------------------------------------------------------
"""


E = 0.5 * kappa * (sin_theta * cos_tilde - cos_theta * sin_tilde)**2
Cl_E01 = Gen.Closure(kappa, F01_q, theta_tilde)

dE01dF01 = VecDiff(E, F01_q)
ddE01ddF01 = VecDiff(dE01dF01, F01_q)

content += f""" // Drving Revolute Joint Energy and Derivatives(Gradient & Hessian)

{Cl_E01("E", E)}
{Cl_E01("dEdF01", dE01dF01)}
{Cl_E01("ddEddF01", ddE01ddF01)}

// ---------
"""

In [4]:
f = open(backend_source_dir('cuda') / 'affine_body/constitutions/sym/affine_body_driving_revolute_joint.inl', 'w')
f.write(content)
f.close()